<a href="https://colab.research.google.com/github/cedizen/detection_automatique_faux_billets/blob/target_model/main_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import joblib
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline

In [19]:
root = "/content/drive/MyDrive/projet_faux_monnayage/"
target = "is_genuine"
file = "data/billets_df_cleaned.csv"

In [34]:
def main():

  # Dataframe given
  df = pd.read_csv(root + file)

  model = joblib.load(f"{root}model.pkl")

  # Check if the model is in the pipeline or not, and extract from it
  if isinstance(model, Pipeline):
    final_estimator = model.steps[-1][1]
  else:
    final_estimator = model

  # Check if the model is unsupervised like KMeans or others
  if isinstance(final_estimator, KMeans):
    clusters = model.predict(df)
    df["clusters"] = clusters
    print(df)
    df.to_csv(f"{root}data/new_csv.csv", index=False)

  # Or supervised and then need to split the target from the dataset
  else:
    # Check if the target is part of the columns
    if target in df.columns:
      X_df = df.drop(target, axis=1)
      y_true = df[target]
    else:
      X_df = df.copy()
      y_true = None

    predictions = model.predict(X_df)
    X_df[target] = y_true
    X_df["predictions"] = predictions

    # calculate the ratio prediction
    if y_true is not None:
      ratio_false_predictions = (y_true != predictions).sum() / len(X_df)
      print(f"{ratio_false_predictions*100:.2f}% of false predictions")

    print(X_df)
    X_df.to_csv(f"{root}data/new_csv.csv", index=False)

In [35]:
if __name__ == "__main__":
  main()

0.87% of false predictions
      diagonal  height_left  height_right  margin_low  margin_up  length  \
0       171.81       104.86        104.95        4.52       2.89  112.83   
1       171.46       103.36        103.66        3.77       2.99  113.09   
2       172.69       104.48        103.50        4.40       2.94  113.16   
3       171.36       103.91        103.94        3.62       3.01  113.51   
4       171.73       104.28        103.46        4.04       3.48  112.54   
...        ...          ...           ...         ...        ...     ...   
1495    171.75       104.38        104.17        4.42       3.09  111.28   
1496    172.19       104.63        104.44        5.27       3.37  110.97   
1497    171.80       104.01        104.12        5.51       3.36  111.95   
1498    172.06       104.28        104.06        5.17       3.46  112.25   
1499    171.47       104.15        103.82        4.63       3.37  112.07   

      is_genuine  predictions  
0           True         Tru